In [ ]:
import os
import base64
import smtplib
from email.mime.text import MIMEText
from html import escape

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

c:\Users\KISHORE S\anaconda3\envs\agent\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
SCOPES = ['https://www.googleapis.com/auth/gmail.send']

In [3]:
def authenticate_gmail():
    creds = None
    
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    
    service = build('gmail', 'v1', credentials=creds)
    return service

In [4]:
def create_message(sender, to, subject, message_text):
    message = MIMEText(message_text)
    message['to'] = to
    message['from'] = sender
    message['subject'] = subject
    
    raw = base64.urlsafe_b64encode(message.as_bytes())
    return {'raw': raw.decode()}

In [5]:
def send_email(service, user_id, message):
    sent_message = service.users().messages().send(
        userId=user_id, body=message).execute()
    print("Message sent! ID:", sent_message['id'])

In [ ]:
service = authenticate_gmail()

sender = "kysh3005@gmail.com"
receiver = "theiitiandrive@gmail.com"

subject = "Test Email from Gmail API"
body = "Hello! This email was sent using Gmail API via Python."

message = create_message(sender, receiver, subject, body)
send_email(service, "me", message)

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=850702669669-cobf0p7m3i0a1c9gpllf8snltlm96ska.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A58964%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.send&state=GtaUT5PnpxZCJ79Fbj5pHXckvT3pO8&code_challenge=bylblZMxJz6PSCjaaaPIie67NG3FHyuLZJ5ZblgpHhg&code_challenge_method=S256&access_type=offline


SyntaxError: invalid syntax (3756711506.py, line 1)

In [ ]:
import smtplib
from email.mime.text import MIMEText


def send_email_smtp(sender_email, app_password, receiver_email, subject, message_text):
    msg = MIMEText(message_text)
    msg["Subject"] = subject
    msg["From"] = sender_email
    msg["To"] = receiver_email

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(sender_email, app_password)
        server.send_message(msg)

    print(f"Email sent successfully to {receiver_email}!")

Email sent successfully!


In [ ]:
def parse_recipient_lines(raw_text):
    recipients = []
    for line in raw_text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        parts = [part.strip() for part in line.split(",")]
        if len(parts) >= 2:
            name = parts[0] or parts[1].split("@")[0].strip()
            email = parts[1]
            notes = ", ".join(parts[2:]).strip() if len(parts) > 2 else ""
        else:
            email = parts[0]
            name = email.split("@")[0].strip()
            notes = ""

        if "@" not in email:
            continue

        recipients.append({
            "name": name,
            "email": email,
            "notes": notes,
        })
    return recipients


def build_email_draft(recipient, sender_name, campaign_subject, campaign_brief, tone, call_to_action):
    name = recipient.get("name") or recipient["email"].split("@")[0]
    notes = recipient.get("notes", "")
    subject = campaign_subject.strip() or f"Hello {name}"

    opener = {
        "Professional": "I hope you are doing well.",
        "Friendly": "I hope you're having a great day.",
        "Concise": "Sharing a quick note.",
        "Persuasive": "I wanted to reach out with a focused opportunity."
    }.get(tone, "I hope you are doing well.")

    body_parts = [
        f"Hi {name},",
        "",
        opener,
        f"{campaign_brief.strip()}",
    ]
    if notes:
        body_parts.extend(["", f"Personal note: {notes}"])
    if call_to_action.strip():
        body_parts.extend(["", call_to_action.strip()])
    if sender_name.strip():
        body_parts.extend(["", f"Best regards,", sender_name.strip()])
    else:
        body_parts.extend(["", "Best regards,"])

    return {
        "name": name,
        "email": recipient["email"],
        "subject": subject,
        "body": "\n".join(body_parts).strip(),
    }


recipients_input = widgets.Textarea(
    value="",
    placeholder="Enter one recipient per line: Name, email, optional notes",
    description="Recipients",
    layout=widgets.Layout(width="100%", height="140px"),
)
sender_email_input = widgets.Text(
    value="",
    placeholder="sender@gmail.com",
    description="Sender email",
    layout=widgets.Layout(width="100%"),
)
app_password_input = widgets.Password(
    value="",
    placeholder="Google App Password",
    description="App password",
    layout=widgets.Layout(width="100%"),
)
sender_name_input = widgets.Text(
    value="",
    placeholder="Your name",
    description="Sender name",
    layout=widgets.Layout(width="100%"),
)
campaign_subject_input = widgets.Text(
    value="",
    placeholder="Subject line for the campaign",
    description="Subject",
    layout=widgets.Layout(width="100%"),
)
campaign_brief_input = widgets.Textarea(
    value="",
    placeholder="Write the core message that should appear in every draft.",
    description="Brief",
    layout=widgets.Layout(width="100%", height="120px"),
)
tone_input = widgets.Dropdown(
    options=["Professional", "Friendly", "Concise", "Persuasive"],
    value="Professional",
    description="Tone",
    layout=widgets.Layout(width="50%"),
)
call_to_action_input = widgets.Textarea(
    value="",
    placeholder="Optional call to action to add to each draft.",
    description="CTA",
    layout=widgets.Layout(width="100%", height="90px"),
)
generate_button = widgets.Button(description="Generate Drafts", button_style="primary")
send_button = widgets.Button(description="Send Approved", button_style="success")
drafts_output = widgets.Output()
status_output = widgets.Output()

email_state = {
    "drafts": [],
    "checkboxes": [],
}


def render_drafts(drafts):
    email_state["drafts"] = drafts
    email_state["checkboxes"] = []

    cards = []
    if not drafts:
        cards.append(widgets.HTML("<b>No drafts generated yet.</b> Add recipients and click <i>Generate Drafts</i>."))
    else:
        for draft in drafts:
            checkbox = widgets.Checkbox(
                value=True,
                description=f"Approve {draft['name']} &lt;{draft['email']}&gt;",
                indent=False,
            )
            email_state["checkboxes"].append(checkbox)
            preview = widgets.HTML(
                f"""
                <div style='border:1px solid #d9d9d9; border-radius:10px; padding:12px; margin:8px 0; background:#fbfbfb;'>
                    <div style='font-weight:700; margin-bottom:6px;'>To: {escape(draft['name'])} &lt;{escape(draft['email'])}&gt;</div>
                    <div style='margin-bottom:8px;'><b>Subject:</b> {escape(draft['subject'])}</div>
                    <pre style='white-space:pre-wrap; margin:0; font-family:inherit;'>{escape(draft['body'])}</pre>
                </div>
                """
            )
            cards.append(widgets.VBox([checkbox, preview]))

    with drafts_output:
        clear_output(wait=True)
        display(widgets.VBox(cards))



def on_generate_clicked(_):
    with status_output:
        clear_output(wait=True)
        recipients = parse_recipient_lines(recipients_input.value)
        if not recipients:
            print("Add at least one recipient in the format: Name, email, optional notes")
            render_drafts([])
            return

        drafts = [
            build_email_draft(
                recipient=recipient,
                sender_name=sender_name_input.value,
                campaign_subject=campaign_subject_input.value,
                campaign_brief=campaign_brief_input.value,
                tone=tone_input.value,
                call_to_action=call_to_action_input.value,
            )
            for recipient in recipients
        ]
        render_drafts(drafts)
        print(f"Generated {len(drafts)} draft(s). Review them in the preview tab and approve the ones you want to send.")



def on_send_clicked(_):
    with status_output:
        clear_output(wait=True)
        drafts = email_state.get("drafts", [])
        checkboxes = email_state.get("checkboxes", [])

        sender_email = sender_email_input.value.strip()
        app_password = app_password_input.value.strip()

        if not sender_email:
            print("Enter the sender email address.")
            return
        if not app_password:
            print("Enter the Google app password.")
            return
        if not drafts:
            print("Generate drafts before sending.")
            return

        approved = [draft for draft, checkbox in zip(drafts, checkboxes) if checkbox.value]
        if not approved:
            print("Select at least one approved draft before sending.")
            return

        for draft in approved:
            send_email_smtp(
                sender_email=sender_email,
                app_password=app_password,
                receiver_email=draft["email"],
                subject=draft["subject"],
                message_text=draft["body"],
            )
        print(f"Sent {len(approved)} approved draft(s).")


generate_button.on_click(on_generate_clicked)
send_button.on_click(on_send_clicked)

generate_panel = widgets.VBox([
    widgets.HTML("<h3>Email Draft Generator</h3><p>Enter recipient details, generate drafts, then review and approve them.</p>"),
    recipients_input,
    sender_name_input,
    campaign_subject_input,
    campaign_brief_input,
    tone_input,
    call_to_action_input,
    generate_button,
    status_output,
])

review_panel = widgets.VBox([
    widgets.HTML("<h3>Approve & Send</h3><p>Review each draft, keep the ones you want, and send only the approved messages.</p>"),
    sender_email_input,
    app_password_input,
    send_button,
    drafts_output,
])

tabs = widgets.Tab(children=[generate_panel, review_panel])
tabs.set_title(0, "Generate Drafts")
tabs.set_title(1, "Approve & Send")
display(tabs)